In [21]:
# ============================================================
# H10: LABEL-AWARE PROTOTYPE + SUPERVISED CONTRASTIVE
#      + CONTROLLED MINORITY LOSS + HD/D SEPARATION
#      + HYPERNETWORK
#
# H10 PURPOSE
# ------------------------------------------------------------
# H10 is the targeted follow-up to H8.
#
# H8:
#   label-aware support
#   prototypes
#   supervised contrastive representation
#   ordinal regularization
#   HyperNetwork
#
# H10 adds:
#
#   1. GENUINELY class-balanced episodic training
#   2. CONTROLLED minority-sensitive CE
#   3. Explicit Highly Disengage vs Disengage separation
#   4. Prototype-based classification signal
#
# IMPORTANT:
#
# TEST DATA IS COMPLETELY UNTOUCHED.
#
# Test support:
#   first K real clips      -> early
#   K random real clips     -> random
#
# Test query:
#   every remaining real clip
#
# NO:
#   synthetic test examples
#   test balancing
#   minority oversampling
#   test class weighting
#
# ------------------------------------------------------------
# IMPORTANT BUG FIX
# ------------------------------------------------------------
#
# Previous H10 error:
#
#   embeddings = 74
#   labels     = 64
#
# This happened because the contrastive embedding tensor was
# constructed from a larger set than the corresponding labels.
#
# H10 below ALWAYS constructs:
#
#       contrastive_embeddings
#       contrastive_labels
#
# from the SAME rows.
#
# Therefore:
#
#       len(embeddings) == len(labels)
#
# is guaranteed before supervised contrastive loss.
# ============================================================


import argparse
import json
import os
import re
import random

from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

NUM_CLASSES = 4

INPUT_DIM = 1024

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(INPUT_DIM)
]


# ============================================================
# MODEL DIMENSIONS
# ============================================================

EMBED_DIM = 128

CONTEXT_DIM = 128

HYPER_HIDDEN = 128

LABEL_EMBED_DIM = 32

ATTENTION_HIDDEN = 128


# ============================================================
# HYPERNETWORK HEAD
# ============================================================

HEAD_PARAM_DIM = (
    NUM_CLASSES * EMBED_DIM
    + NUM_CLASSES
)


# ============================================================
# TRAINING
# ============================================================

DEFAULT_EPISODES = 2000

DEFAULT_SEED = 42

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 5.0


# ============================================================
# H10 LOSS PARAMETERS
# ============================================================

# Supervised contrastive temperature
CONTRASTIVE_TEMPERATURE = 0.07

# Weight of supervised contrastive loss
LAMBDA_CONTRASTIVE = 0.10

# Weight of prototype classification loss
LAMBDA_PROTOTYPE = 0.20

# Weight of ordinal regularization
LAMBDA_ORDINAL = 0.05

# Weight of explicit HD/D separation
LAMBDA_HD_D = 0.10

# Controlled minority-sensitive CE.
#
# This is deliberately NOT aggressive.
#
# Class weights:
#
# HD = 2.0
# D  = 1.25
# E  = 1.0
# HE = 1.25
#
# This prevents Engage from completely dominating while
# avoiding the instability of huge inverse-frequency weights.
#
MINORITY_CLASS_WEIGHTS = torch.tensor(
    [
        2.0,    # Highly Disengage
        1.25,   # Disengage
        1.0,    # Engage
        1.25,   # Highly Engage
    ],
    dtype=torch.float32,
)


# ============================================================
# TRAINING SUPPORT
# ============================================================

TRAIN_K_VALUES = [
    5,
    10,
    20,
]

# Maximum query samples used by the episodic objective.
#
# IMPORTANT:
#
# The contrastive branch does NOT blindly use this value.
# It uses the exact query tensor and exact query labels.
#
QUERY_MAX = 64


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(
    clip_id: str
):

    """
    Expected:

        video123_17_person4

    becomes:

        video123_person4
    """

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id),
    )

    if not match:

        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(
    json_path
):

    with open(
        json_path,
        "r",
        encoding="utf-8",
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(
            meta,
            dict,
        ):

            continue

        label = meta.get(
            "label"
        )

        if label not in LABELS:

            continue

        embeds = meta.get(
            "embeds"
        )

        if not isinstance(
            embeds,
            list,
        ):

            continue

        if len(embeds) != INPUT_DIM:

            continue

        group = (
            parse_group_from_clip_id(
                clip_id
            )
        )

        if group is None:

            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32,
        )

        if not np.all(
            np.isfinite(arr)
        ):

            continue

        match = re.search(
            r"_(\d+)_",
            str(clip_id),
        )

        clip_index = (
            int(match.group(1))
            if match
            else 0
        )

        records.append(
            {
                "clip_id": str(
                    clip_id
                ),

                "group": group,

                "label": str(
                    label
                ),

                "label_id":
                    LABEL_TO_ID[
                        str(label)
                    ],

                "clip_index":
                    clip_index,

                "embedding":
                    arr,
            }
        )

    if not records:

        raise ValueError(
            f"No valid labelled clips "
            f"found in {json_path}"
        )

    df = pd.DataFrame(
        records
    )

    emb_matrix = np.vstack(
        df[
            "embedding"
        ].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=[
                    "embedding"
                ]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


# ============================================================
# FOLD MAP
# ============================================================

def load_group_fold_map(
    fold_csv_path
):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        - set(fold_df.columns)
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            [
                "group",
                "fold",
            ]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby("group")[
            "fold"
        ]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            f"folds: {bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold["group"],
            group_fold["fold"].astype(int),
        )
    )


# ============================================================
# TEST SUPPORT / QUERY SAMPLING
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):

    """
    COMPLETELY UNTOUCHED TEST PROTOCOL.

    early:
        first K real clips

    random:
        K random real clips

    query:
        all remaining real clips

    NO balancing.
    NO synthetic data.
    NO class manipulation.
    """

    ordered = (
        student_df
        .sort_values(
            "clip_index"
        )
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            (
                f"insufficient_clips_n="
                f"{len(ordered)}_k="
                f"{k_shot}"
            ),
        )

    if protocol == "early":

        support = ordered.iloc[
            :k_shot
        ].copy()

    elif protocol == "random":

        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = ordered.loc[
            sampled_idx
        ].copy()

    else:

        return (
            None,
            None,
            (
                f"unknown_protocol_"
                f"{protocol}"
            ),
        )

    query = ordered.loc[
        ~ordered.index.isin(
            support.index
        )
    ].copy()

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# CLASS-BALANCED TRAINING SUPPORT
# ============================================================

def sample_balanced_support(
    student_df,
    k_shot,
    rng,
):

    """
    TRAINING ONLY.

    Attempts to distribute support slots across all
    classes available for the student.
    """

    class_indices = {}

    available_classes = []

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            student_df[
                student_df[
                    "label_id"
                ]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:

        return None

    n_classes = len(
        available_classes
    )

    base = (
        k_shot
        // n_classes
    )

    remainder = (
        k_shot
        % n_classes
    )

    allocation = {
        c: base
        for c in available_classes
    }

    shuffled = (
        available_classes.copy()
    )

    rng.shuffle(
        shuffled
    )

    for c in shuffled[
        :remainder
    ]:

        allocation[c] += 1

    selected = []

    unused = 0

    for c in available_classes:

        requested = allocation[c]

        available = class_indices[c]

        take = min(
            requested,
            len(available),
        )

        unused += (
            requested - take
        )

        if take > 0:

            chosen = rng.choice(
                available,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # --------------------------------------------------------
    # Redistribute unused slots
    # --------------------------------------------------------

    if unused > 0:

        selected_set = set(
            selected
        )

        remaining = []

        for c in available_classes:

            for idx in class_indices[c]:

                if idx not in selected_set:

                    remaining.append(
                        idx
                    )

        if remaining:

            take = min(
                unused,
                len(remaining),
            )

            extra = rng.choice(
                np.asarray(
                    remaining
                ),
                size=take,
                replace=False,
            )

            selected.extend(
                extra.tolist()
            )

    if not selected:

        return None

    return student_df.loc[
        selected
    ].copy()


# ============================================================
# CLASS-BALANCED TRAINING QUERY
# ============================================================

def sample_balanced_query(
    query_df,
    max_samples,
    rng,
):

    """
    TRAINING ONLY.

    Produces a class-balanced query batch.

    This function NEVER appears in evaluation.
    """

    if query_df.empty:

        return query_df

    class_indices = {}

    available_classes = []

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            query_df[
                query_df[
                    "label_id"
                ]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if not available_classes:

        return query_df.iloc[0:0]

    n_classes = len(
        available_classes
    )

    base = (
        max_samples
        // n_classes
    )

    remainder = (
        max_samples
        % n_classes
    )

    allocation = {
        c: base
        for c in available_classes
    }

    shuffled = (
        available_classes.copy()
    )

    rng.shuffle(
        shuffled
    )

    for c in shuffled[
        :remainder
    ]:

        allocation[c] += 1

    selected = []

    for c in available_classes:

        idx = class_indices[c]

        take = min(
            allocation[c],
            len(idx),
        )

        if take > 0:

            chosen = rng.choice(
                idx,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # --------------------------------------------------------
    # Fill remaining slots
    # --------------------------------------------------------

    selected_set = set(
        selected
    )

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(
                    idx
                )

    slots = (
        max_samples
        - len(selected)
    )

    if (
        slots > 0
        and remaining
    ):

        take = min(
            slots,
            len(remaining),
        )

        extra = rng.choice(
            np.asarray(
                remaining
            ),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    return query_df.loc[
        selected
    ].copy()


# ============================================================
# LABEL-AWARE SUPPORT ENCODER
# ============================================================

class LabelAwareContextEncoder(
    nn.Module
):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
        context_dim=CONTEXT_DIM,
        label_embed_dim=LABEL_EMBED_DIM,
        attention_hidden=ATTENTION_HIDDEN,
    ):

        super().__init__()

        self.embedding_encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                embed_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                embed_dim
            ),
        )

        self.label_embedding = (
            nn.Embedding(
                NUM_CLASSES,
                label_embed_dim,
            )
        )

        self.support_fusion = nn.Sequential(

            nn.Linear(
                embed_dim
                + label_embed_dim,
                context_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                context_dim
            ),
        )

        self.attention = nn.Sequential(

            nn.Linear(
                context_dim,
                attention_hidden,
            ),

            nn.Tanh(),

            nn.Linear(
                attention_hidden,
                1,
            ),
        )

        self.context_projection = nn.Sequential(

            nn.Linear(
                context_dim,
                context_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                context_dim
            ),
        )

    def encode_support(
        self,
        support_x,
        support_y,
    ):

        visual = (
            self.embedding_encoder(
                support_x
            )
        )

        labels = (
            self.label_embedding(
                support_y
            )
        )

        fused = torch.cat(
            [
                visual,
                labels,
            ],
            dim=1,
        )

        support_repr = (
            self.support_fusion(
                fused
            )
        )

        attention_logits = (
            self.attention(
                support_repr
            )
        )

        attention_weights = (
            F.softmax(
                attention_logits,
                dim=0,
            )
        )

        context = (
            attention_weights
            * support_repr
        ).sum(
            dim=0
        )

        context = (
            self.context_projection(
                context
            )
        )

        return (
            support_repr,
            context,
            attention_weights.squeeze(
                -1
            ),
        )

    def forward(
        self,
        support_x,
        support_y,
    ):

        (
            support_repr,
            context,
            attention,
        ) = self.encode_support(
            support_x,
            support_y,
        )

        return (
            context,
            attention,
            support_repr,
        )


# ============================================================
# QUERY ENCODER
# ============================================================

class QueryEncoder(
    nn.Module
):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                256,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                256
            ),

            nn.Linear(
                256,
                embed_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                embed_dim
            ),
        )

    def forward(
        self,
        x,
    ):

        return self.net(x)


# ============================================================
# HYPERNETWORK
# ============================================================

class HyperNetwork(
    nn.Module
):

    def __init__(
        self,
        context_dim=CONTEXT_DIM,
        hidden_dim=HYPER_HIDDEN,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.embed_dim = (
            embed_dim
        )

        self.net = nn.Sequential(

            nn.Linear(
                context_dim,
                hidden_dim,
            ),

            nn.ReLU(),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),

            nn.ReLU(),
        )

        self.output = nn.Linear(
            hidden_dim,
            HEAD_PARAM_DIM,
        )

    def forward(
        self,
        context,
    ):

        params = self.output(
            self.net(context)
        )

        weight_end = (
            NUM_CLASSES
            * self.embed_dim
        )

        weight = (
            params[
                :weight_end
            ]
            .view(
                NUM_CLASSES,
                self.embed_dim,
            )
        )

        bias = params[
            weight_end:
        ]

        return (
            weight,
            bias,
        )


# ============================================================
# PROTOTYPE MODULE
# ============================================================

class PrototypeClassifier(
    nn.Module
):

    """
    Explicit class prototypes.

    Each available support class gets a prototype.

    IMPORTANT:

    If a class is not present in support, it is NOT fabricated.

    The prototype mask records which classes are genuinely
    available.
    """

    def __init__(
        self,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.embed_dim = (
            embed_dim
        )

    def forward(
        self,
        support_embeddings,
        support_labels,
        query_embeddings,
    ):

        prototypes = []

        available = []

        for class_id in range(
            NUM_CLASSES
        ):

            mask = (
                support_labels
                == class_id
            )

            if mask.any():

                prototype = (
                    support_embeddings[
                        mask
                    ]
                    .mean(
                        dim=0
                    )
                )

                prototype = F.normalize(
                    prototype,
                    dim=0,
                )

                prototypes.append(
                    prototype
                )

                available.append(
                    True
                )

            else:

                # Dummy tensor only for shape.
                #
                # It is masked out later and is NEVER treated
                # as a real prototype.
                prototypes.append(
                    torch.zeros(
                        self.embed_dim,
                        device=(
                            support_embeddings
                            .device
                        ),
                    )
                )

                available.append(
                    False
                )

        prototypes = torch.stack(
            prototypes,
            dim=0,
        )

        available = torch.tensor(
            available,
            dtype=torch.bool,
            device=(
                support_embeddings.device
            ),
        )

        query_norm = F.normalize(
            query_embeddings,
            dim=1,
        )

        similarities = (
            query_norm
            @ prototypes.T
        )

        # Mask unavailable prototypes.
        similarities = similarities.masked_fill(
            ~available.unsqueeze(0),
            -1e9,
        )

        return (
            similarities,
            prototypes,
            available,
        )


# ============================================================
# FULL H10 MODEL
# ============================================================

class H10HyperNetworkModel(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.context_encoder = (
            LabelAwareContextEncoder()
        )

        self.query_encoder = (
            QueryEncoder()
        )

        self.hypernetwork = (
            HyperNetwork()
        )

        self.prototype_classifier = (
            PrototypeClassifier()
        )

    def generate_student_head(
        self,
        support_x,
        support_y,
    ):

        (
            context,
            attention,
            support_repr,
        ) = self.context_encoder(
            support_x,
            support_y,
        )

        weight, bias = (
            self.hypernetwork(
                context
            )
        )

        return (
            weight,
            bias,
            context,
            attention,
            support_repr,
        )

    def forward_episode(
        self,
        support_x,
        support_y,
        query_x,
    ):

        (
            weight,
            bias,
            context,
            attention,
            support_repr,
        ) = self.generate_student_head(
            support_x,
            support_y,
        )

        query_emb = (
            self.query_encoder(
                query_x
            )
        )

        logits = F.linear(
            query_emb,
            weight,
            bias,
        )

        (
            prototype_logits,
            prototypes,
            prototype_available,
        ) = self.prototype_classifier(
            support_repr,
            support_y,
            query_emb,
        )

        return {
            "logits":
                logits,

            "query_embeddings":
                query_emb,

            "support_embeddings":
                support_repr,

            "context":
                context,

            "attention":
                attention,

            "prototype_logits":
                prototype_logits,

            "prototypes":
                prototypes,

            "prototype_available":
                prototype_available,
        }

    def predict(
        self,
        support_x,
        support_y,
        query_x,
    ):

        return self.forward_episode(
            support_x,
            support_y,
            query_x,
        )


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

def supervised_contrastive_loss(
    embeddings,
    labels,
    temperature=CONTRASTIVE_TEMPERATURE,
):

    """
    STANDARD SUPERVISED CONTRASTIVE LOSS.

    CRITICAL H10 FIX:

    embeddings and labels MUST correspond to the SAME rows.

    We explicitly validate:

        embeddings.shape[0]
        ==
        labels.shape[0]

    before constructing masks.

    This prevents the previous:

        embeddings=74
        labels=64

    crash.
    """

    if embeddings.ndim != 2:

        raise ValueError(
            "Contrastive embeddings "
            "must have shape [N, D]. "
            f"Got {embeddings.shape}"
        )

    n = embeddings.shape[0]

    if labels.ndim != 1:

        labels = labels.view(-1)

    if labels.shape[0] != n:

        raise ValueError(
            "Contrastive loss label/embedding "
            "mismatch: "
            f"embeddings={n}, "
            f"labels={labels.shape[0]}"
        )

    if n < 2:

        return embeddings.sum() * 0.0

    embeddings = F.normalize(
        embeddings,
        dim=1,
    )

    similarity = (
        torch.matmul(
            embeddings,
            embeddings.T,
        )
        / temperature
    )

    # --------------------------------------------------------
    # Remove self similarity
    # --------------------------------------------------------

    logits_mask = (
        torch.ones(
            n,
            n,
            device=embeddings.device,
        )
        - torch.eye(
            n,
            device=embeddings.device,
        )
    )

    # --------------------------------------------------------
    # Positive mask
    # --------------------------------------------------------

    labels = labels.contiguous()

    positive_mask = (
        labels.unsqueeze(0)
        == labels.unsqueeze(1)
    ).float()

    positive_mask = (
        positive_mask
        * logits_mask
    )

    # --------------------------------------------------------
    # Numerical stabilization
    # --------------------------------------------------------

    logits = similarity

    logits = logits.masked_fill(
        logits_mask == 0,
        -1e9,
    )

    log_prob = (
        logits
        - torch.logsumexp(
            logits,
            dim=1,
            keepdim=True,
        )
    )

    positive_count = (
        positive_mask.sum(
            dim=1
        )
    )

    valid = (
        positive_count > 0
    )

    if not valid.any():

        return embeddings.sum() * 0.0

    mean_log_prob_pos = (
        (
            positive_mask
            * log_prob
        )
        .sum(dim=1)
        / positive_count.clamp(
            min=1.0
        )
    )

    loss = -mean_log_prob_pos[
        valid
    ].mean()

    return loss


# ============================================================
# CONTROLLED MINORITY CE
# ============================================================

def minority_sensitive_ce(
    logits,
    targets,
):

    weights = (
        MINORITY_CLASS_WEIGHTS
        .to(logits.device)
    )

    return F.cross_entropy(
        logits,
        targets,
        weight=weights,
    )


# ============================================================
# ORDINAL REGULARIZATION
# ============================================================

def ordinal_regularization(
    logits,
    targets,
):

    """
    Encourages neighboring engagement states to have
    approximately neighboring scores.

    Classes:

        0 HD
        1 D
        2 E
        3 HE

    We compute expected ordinal position and penalize large
    deviation from the target ordinal value.

    This is intentionally small in H10.
    """

    probabilities = F.softmax(
        logits,
        dim=1,
    )

    class_positions = (
        torch.arange(
            NUM_CLASSES,
            dtype=torch.float32,
            device=logits.device,
        )
    )

    expected_position = (
        probabilities
        * class_positions.unsqueeze(0)
    ).sum(
        dim=1
    )

    target_position = (
        targets.float()
    )

    return F.smooth_l1_loss(
        expected_position,
        target_position,
    )


# ============================================================
# EXPLICIT HD / D PROTOTYPE SEPARATION
# ============================================================

def hd_d_separation_loss(
    prototypes,
    available,
):

    """
    Explicitly separates:

        Highly Disengage
        Disengage

    prototype vectors.

    We use cosine similarity.

    Desired behavior:

        similarity(HD, D)

    should be lower.

    The margin is deliberately moderate.
    """

    if not (
        available[0]
        and available[1]
    ):

        return prototypes.sum() * 0.0

    hd = F.normalize(
        prototypes[0],
        dim=0,
    )

    d = F.normalize(
        prototypes[1],
        dim=0,
    )

    similarity = (
        torch.sum(
            hd * d
        )
    )

    # Margin is deliberately modest.
    margin = 0.20

    return F.relu(
        similarity - margin
    )


# ============================================================
# TRAINING PROTOTYPE LOSS
# ============================================================

def prototype_classification_loss(
    prototype_logits,
    targets,
):

    """
    Prototype classification loss.

    Only valid when the target's prototype is available.

    Since training support is class-aware, this should usually
    be valid for the classes represented in the episode.
    """

    valid = torch.zeros_like(
        targets,
        dtype=torch.bool,
    )

    # A prototype is available whenever the corresponding
    # logit is not the artificial -1e9 mask.
    target_logits = (
        prototype_logits[
            torch.arange(
                len(targets),
                device=targets.device,
            ),
            targets,
        ]
    )

    valid = (
        target_logits > -1e8
    )

    if not valid.any():

        return prototype_logits.sum() * 0.0

    return F.cross_entropy(
        prototype_logits[valid],
        targets[valid],
    )


# ============================================================
# H10 TOTAL LOSS
# ============================================================

def h10_total_loss(
    outputs,
    query_y,
):

    logits = outputs[
        "logits"
    ]

    prototype_logits = outputs[
        "prototype_logits"
    ]

    query_embeddings = outputs[
        "query_embeddings"
    ]

    # --------------------------------------------------------
    # Main controlled minority-sensitive CE
    # --------------------------------------------------------

    loss_ce = (
        minority_sensitive_ce(
            logits,
            query_y,
        )
    )

    # --------------------------------------------------------
    # Prototype classification
    # --------------------------------------------------------

    loss_proto = (
        prototype_classification_loss(
            prototype_logits,
            query_y,
        )
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Contrastive embeddings and labels come from EXACTLY
    # the same query rows.
    #
    # No support embeddings are concatenated here.
    #
    # This is the direct fix for the previous:
    #
    #   embeddings=74
    #   labels=64
    #
    # mismatch.
    # --------------------------------------------------------

    contrastive_embeddings = (
        query_embeddings
    )

    contrastive_labels = (
        query_y
    )

    if (
        contrastive_embeddings.shape[0]
        != contrastive_labels.shape[0]
    ):

        raise RuntimeError(
            "INTERNAL H10 BUG: contrastive "
            "embedding/label mismatch before "
            "loss calculation: "
            f"embeddings="
            f"{contrastive_embeddings.shape[0]}, "
            f"labels="
            f"{contrastive_labels.shape[0]}"
        )

    loss_contrastive = (
        supervised_contrastive_loss(
            contrastive_embeddings,
            contrastive_labels,
        )
    )

    # --------------------------------------------------------
    # Ordinal
    # --------------------------------------------------------

    loss_ordinal = (
        ordinal_regularization(
            logits,
            query_y,
        )
    )

    # --------------------------------------------------------
    # HD/D separation
    # --------------------------------------------------------

    loss_hd_d = (
        hd_d_separation_loss(
            outputs[
                "prototypes"
            ],
            outputs[
                "prototype_available"
            ],
        )
    )

    # --------------------------------------------------------
    # Total
    # --------------------------------------------------------

    total = (
        loss_ce

        + LAMBDA_PROTOTYPE
        * loss_proto

        + LAMBDA_CONTRASTIVE
        * loss_contrastive

        + LAMBDA_ORDINAL
        * loss_ordinal

        + LAMBDA_HD_D
        * loss_hd_d
    )

    loss_dict = {

        "total":
            total,

        "ce":
            loss_ce,

        "prototype":
            loss_proto,

        "contrastive":
            loss_contrastive,

        "ordinal":
            loss_ordinal,

        "hd_d":
            loss_hd_d,
    }

    return (
        total,
        loss_dict,
    )


# ============================================================
# EPISODIC TRAINING
# ============================================================

def episodic_train(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df[
            "group"
        ].unique()
    )

    if not groups:

        raise ValueError(
            "No training students available."
        )

    successful = 0

    running_losses = []

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Sample student
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = (
            train_df[
                train_df[
                    "group"
                ]
                == group
            ]
            .copy()
        )

        # ----------------------------------------------------
        # K
        # ----------------------------------------------------

        k = int(
            rng.choice(
                TRAIN_K_VALUES
            )
        )

        if len(student_df) <= k:

            continue

        # ----------------------------------------------------
        # CLASS-BALANCED SUPPORT
        # ----------------------------------------------------

        support_df = (
            sample_balanced_support(
                student_df,
                k,
                rng,
            )
        )

        if (
            support_df is None
            or support_df.empty
        ):

            continue

        # ----------------------------------------------------
        # Remaining query pool
        # ----------------------------------------------------

        query_pool = student_df.loc[
            ~student_df.index.isin(
                support_df.index
            )
        ].copy()

        if query_pool.empty:

            continue

        # ----------------------------------------------------
        # CLASS-BALANCED QUERY
        # ----------------------------------------------------

        query_df = (
            sample_balanced_query(
                query_pool,
                QUERY_MAX,
                rng,
            )
        )

        if query_df.empty:

            continue

        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        support_x_np = scaler.transform(
            support_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_x_np = scaler.transform(
            query_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        support_y_np = (
            support_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        query_y_np = (
            query_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        # ----------------------------------------------------
        # Torch tensors
        # ----------------------------------------------------

        support_x = torch.tensor(
            support_x_np,
            dtype=torch.float32,
            device=device,
        )

        support_y = torch.tensor(
            support_y_np,
            dtype=torch.long,
            device=device,
        )

        query_x = torch.tensor(
            query_x_np,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y_np,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # HARD SAFETY CHECK
        #
        # These are the exact tensors passed into contrastive
        # learning.
        # ----------------------------------------------------

        if (
            query_x.shape[0]
            != query_y.shape[0]
        ):

            raise RuntimeError(
                "H10 query tensor mismatch: "
                f"query_x="
                f"{query_x.shape[0]}, "
                f"query_y="
                f"{query_y.shape[0]}"
            )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = (
            model.forward_episode(
                support_x,
                support_y,
                query_x,
            )
        )

        # ----------------------------------------------------
        # Another explicit contrastive safety check
        # ----------------------------------------------------

        if (
            outputs[
                "query_embeddings"
            ].shape[0]
            != query_y.shape[0]
        ):

            raise RuntimeError(
                "H10 query embedding/label "
                "mismatch: "
                f"query_embeddings="
                f"{outputs['query_embeddings'].shape[0]}, "
                f"query_y="
                f"{query_y.shape[0]}"
            )

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        (
            loss,
            loss_dict,
        ) = h10_total_loss(
            outputs,
            query_y,
        )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP,
        )

        optimizer.step()

        successful += 1

        running_losses.append(
            float(
                loss.item()
            )
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (
            (episode + 1) % 100
            == 0
        ):

            recent = (
                running_losses[
                    -100:
                ]
            )

            print(
                f"    Episode "
                f"{episode + 1}/"
                f"{episodes} "
                f"loss="
                f"{np.mean(recent):.4f} "
                f"CE="
                f"{loss_dict['ce'].item():.4f} "
                f"Proto="
                f"{loss_dict['prototype'].item():.4f} "
                f"SupCon="
                f"{loss_dict['contrastive'].item():.4f} "
                f"Ord="
                f"{loss_dict['ordinal'].item():.4f} "
                f"HD/D="
                f"{loss_dict['hd_d'].item():.4f}"
            )

    print(
        f"    Successful training "
        f"episodes: "
        f"{successful}/"
        f"{episodes}"
    )

    if running_losses:

        print(
            f"    Final mean loss: "
            f"{np.mean(running_losses[-100:]):.4f}"
        )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {

        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(
                balanced_accuracy
            ),

        "macro_f1":
            float(macro_f1),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support,

        "confusion_matrix":
            cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query,
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get,
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# PROTOTYPE PREDICTION
# ============================================================

def prototype_predictions(
    prototype_logits,
):

    return (
        torch.argmax(
            prototype_logits,
            dim=1,
        )
        .cpu()
        .numpy()
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    support_x_np = scaler.transform(
        support_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    query_x_np = scaler.transform(
        query_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    support_y_np = (
        support_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_true = (
        query_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    support_x = torch.tensor(
        support_x_np,
        dtype=torch.float32,
        device=device,
    )

    support_y = torch.tensor(
        support_y_np,
        dtype=torch.long,
        device=device,
    )

    query_x = torch.tensor(
        query_x_np,
        dtype=torch.float32,
        device=device,
    )

    model.eval()

    with torch.no_grad():

        outputs = (
            model.predict(
                support_x,
                support_y,
                query_x,
            )
        )

        logits = outputs[
            "logits"
        ]

        prototype_logits = outputs[
            "prototype_logits"
        ]

        y_pred = (
            torch.argmax(
                logits,
                dim=1,
            )
            .cpu()
            .numpy()
        )

        prototype_pred = (
            prototype_predictions(
                prototype_logits
            )
        )

        prototype_available = (
            outputs[
                "prototype_available"
            ]
            .cpu()
            .numpy()
        )

        context_norm = (
            outputs[
                "context"
            ]
            .norm()
            .item()
        )

    metrics = calculate_metrics(
        y_true,
        y_pred,
    )

    prototype_metrics = (
        calculate_metrics(
            y_true,
            prototype_pred,
        )
    )

    majority_pred = (
        majority_prediction(
            y_true
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_true,
            majority_pred,
        )
    )

    return {
        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "prototype_pred":
            prototype_pred,

        "metrics":
            metrics,

        "prototype_metrics":
            prototype_metrics,

        "majority_metrics":
            majority_metrics,

        "prototype_available":
            prototype_available,

        "context_norm":
            context_norm,
    }


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df[
            "group"
        ].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []

    per_student_rows = []

    per_class_rows = []

    confusion_rows = []

    skipped_rows = []

    # ========================================================
    # OUTER CV
    # ========================================================

    fold_ids = sorted(
        set(
            group_fold.values()
        )
    )

    for fold_id in fold_ids:

        print()
        print("=" * 80)

        print(
            f"H10 OUTER FOLD "
            f"{fold_id}"
        )

        print("=" * 80)

        test_groups = [
            g
            for g, f
            in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f
            in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df[
                "group"
            ].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df[
                "group"
            ].isin(
                test_groups
            )
        ].copy()

        # ----------------------------------------------------
        # Leakage check
        # ----------------------------------------------------

        overlap = (
            set(
                train_df[
                    "group"
                ]
            )
            &
            set(
                test_df[
                    "group"
                ]
            )
        )

        if overlap:

            raise RuntimeError(
                f"DATA LEAKAGE in fold "
                f"{fold_id}: "
                f"{overlap}"
            )

        print(
            f"Training students: "
            f"{train_df['group'].nunique()}"
        )

        print(
            f"Test students: "
            f"{test_df['group'].nunique()}"
        )

        print()

        print(
            "Training class distribution:"
        )

        print(
            train_df[
                "label"
            ].value_counts()
        )

        # ----------------------------------------------------
        # Scaler fit ONLY on training students
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train = (
            train_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        scaler.fit(
            X_train
        )

        # ----------------------------------------------------
        # Fresh model
        # ----------------------------------------------------

        set_seed(
            seed + fold_id
        )

        model = (
            H10HyperNetworkModel()
        )

        model.to(device)

        print()

        print(
            f"H10 meta-training "
            f"({episodes} episodes)"
        )

        episodic_train(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                + fold_id * 1000
            ),
            device=device,
        )

        # ====================================================
        # TEST STUDENTS
        # ====================================================

        for group in test_groups:

            student_df = test_df[
                test_df[
                    "group"
                ]
                == group
            ].copy()

            if student_df.empty:

                continue

            for k in k_values:

                for protocol in [
                    "random",
                    "early",
                ]:

                    local_seed = (
                        seed
                        + fold_id * 10000
                        + k * 100
                        + sum(
                            ord(ch)
                            for ch
                            in group
                        )
                        + (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = sample_support_query(
                        student_df,
                        k,
                        protocol,
                        rng,
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {
                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "reason":
                                    reason,
                            }
                        )

                        continue

                    # ------------------------------------------------
                    # REAL TEST SUPPORT.
                    #
                    # ABSOLUTELY NO BALANCING.
                    # ------------------------------------------------

                    result = (
                        evaluate_student(
                            model=model,
                            support_df=support_df,
                            query_df=query_df,
                            scaler=scaler,
                            device=device,
                        )
                    )

                    metrics = result[
                        "metrics"
                    ]

                    prototype_metrics = result[
                        "prototype_metrics"
                    ]

                    majority_metrics = result[
                        "majority_metrics"
                    ]

                    prototype_available = result[
                        "prototype_available"
                    ]

                    # ------------------------------------------------
                    # Student-level result
                    # ------------------------------------------------

                    per_student_rows.append(
                        {

                            "method":
                                "H10_"
                                "label_aware_"
                                "prototype_"
                                "supcon_"
                                "minority_"
                                "ordinal_"
                                "hypernetwork",

                            "outer_fold":
                                fold_id,

                            "student_group":
                                group,

                            "k_shot":
                                k,

                            "support_protocol":
                                protocol,

                            "support_size":
                                len(
                                    support_df
                                ),

                            "query_size":
                                len(
                                    query_df
                                ),

                            "accuracy":
                                metrics[
                                    "accuracy"
                                ],

                            "balanced_accuracy":
                                metrics[
                                    "balanced_accuracy"
                                ],

                            "macro_f1":
                                metrics[
                                    "macro_f1"
                                ],

                            "hyper_accuracy":
                                metrics[
                                    "accuracy"
                                ],

                            "prototype_accuracy":
                                prototype_metrics[
                                    "accuracy"
                                ],

                            "prototype_balanced_accuracy":
                                prototype_metrics[
                                    "balanced_accuracy"
                                ],

                            "majority_accuracy":
                                majority_metrics[
                                    "accuracy"
                                ],

                            "majority_balanced_accuracy":
                                majority_metrics[
                                    "balanced_accuracy"
                                ],

                            "majority_macro_f1":
                                majority_metrics[
                                    "macro_f1"
                                ],

                            "prototype_classes_available":
                                int(
                                    prototype_available.sum()
                                ),

                            "context_norm":
                                result[
                                    "context_norm"
                                ],
                        }
                    )

                    # ------------------------------------------------
                    # Per-class
                    # ------------------------------------------------

                    for class_id, label in enumerate(
                        LABELS
                    ):

                        per_class_rows.append(
                            {

                                "method":
                                    "H10",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][class_id]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][class_id]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][class_id]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][class_id]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {

                                    "method":
                                        "H10",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

        print(
            f"Completed fold "
            f"{fold_id}"
        )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = pd.DataFrame(
        per_student_rows
    )

    per_class_df = pd.DataFrame(
        per_class_rows
    )

    confusion_df = pd.DataFrame(
        confusion_rows
    )

    skipped_df = pd.DataFrame(
        skipped_rows
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No student evaluations "
            "were produced."
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_hyper_accuracy=(
                "hyper_accuracy",
                "mean",
            ),

            mean_prototype_accuracy=(
                "prototype_accuracy",
                "mean",
            ),

            mean_prototype_balanced_accuracy=(
                "prototype_balanced_accuracy",
                "mean",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),

            mean_prototype_classes_available=(
                "prototype_classes_available",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # CLASS SUMMARY
    # ========================================================

    class_summary_df = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    # ========================================================
    # OUTPUT
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "h10_per_student.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "h10_summary.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "h10_per_class.csv",
    )

    class_summary_path = os.path.join(
        output_dir,
        "h10_class_summary.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "h10_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "h10_skipped.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    per_class_df.to_csv(
        per_class_path,
        index=False,
    )

    class_summary_df.to_csv(
        class_summary_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print()
    print("=" * 80)

    print(
        "H10: RESULTS"
    )

    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # PER CLASS
    # ========================================================

    print()
    print("=" * 80)

    print(
        "H10 PER-CLASS RESULTS"
    )

    print("=" * 80)

    print(
        class_summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # HIGHLY DISENGAGE
    # ========================================================

    print()
    print("=" * 80)

    print(
        "H10 HIGHLY DISENGAGE RESULTS"
    )

    print("=" * 80)

    hd_results = per_class_df[
        per_class_df[
            "class"
        ]
        == "Highly Disengage"
    ]

    if hd_results.empty:

        print(
            "No Highly Disengage "
            "evaluation rows."
        )

    else:

        print(
            hd_results
            .groupby(
                [
                    "support_protocol",
                    "k_shot",
                ],
                as_index=False,
            )
            .agg(

                mean_precision=(
                    "precision",
                    "mean",
                ),

                mean_recall=(
                    "recall",
                    "mean",
                ),

                mean_f1=(
                    "f1",
                    "mean",
                ),

                total_support=(
                    "support",
                    "sum",
                ),
            )
            .to_string(
                index=False
            )
        )

    # ========================================================
    # SKIPPED
    # ========================================================

    print()
    print("=" * 80)

    print(
        "H10 SKIPPED CASES"
    )

    print("=" * 80)

    if skipped_df.empty:

        print(
            "None."
        )

    else:

        print(
            skipped_df[
                "reason"
            ].value_counts()
        )

    # ========================================================
    # SAVED FILES
    # ========================================================

    print()
    print(
        "Saved:"
    )

    print(
        per_student_path
    )

    print(
        summary_path
    )

    print(
        per_class_path
    )

    print(
        class_summary_path
    )

    print(
        confusion_path
    )

    print(
        skipped_path
    )

    return (
        per_student_df,
        summary_df,
        per_class_df,
        class_summary_df,
        confusion_df,
        skipped_df,
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description=(
            "H10 label-aware prototype "
            "+ supervised contrastive "
            "+ controlled minority loss "
            "+ ordinal HyperNetwork."
        )
    )

    parser.add_argument(
        "--json",
        type=str,
        default=(
            "/content/final_data_1.json"
        ),
    )

    parser.add_argument(
        "--folds",
        type=str,
        default=(
            "/content/"
            "cmose_h1_h2_fold_protocol.csv"
        ),
    )

    parser.add_argument(
        "--k-values",
        type=str,
        default="5,10,20",
    )

    parser.add_argument(
        "--episodes",
        type=int,
        default=DEFAULT_EPISODES,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=DEFAULT_SEED,
    )

    parser.add_argument(
        "--output-dir",
        type=str,
        default="/content/h10_results",
    )

    parser.add_argument(
        "--device",
        type=str,
        default="auto",
        choices=[
            "auto",
            "cpu",
            "cuda",
        ],
    )

    args = parser.parse_args()

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if args.device == "cuda":

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA requested but unavailable."
            )

        device = torch.device(
            "cuda"
        )

    elif args.device == "cpu":

        device = torch.device(
            "cpu"
        )

    else:

        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(
        f"Using device: {device}"
    )

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    set_seed(
        args.seed
    )

    k_values = [
        int(v.strip())
        for v in args.k_values.split(",")
        if v.strip()
    ]

    clip_df = (
        load_clip_dataframe(
            args.json
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()

    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    group_fold = (
        load_group_fold_map(
            args.folds
        )
    )

    print()

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=k_values,
        seed=args.seed,
        episodes=args.episodes,
        device=device,
        output_dir=args.output_dir,
    )


# ============================================================
# COLAB ENTRY POINT
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # DIRECT COLAB CONFIGURATION
    # --------------------------------------------------------

    JSON_PATH = (
        "/content/final_data_1.json"
    )

    FOLDS_PATH = (
        "/content/"
        "cmose_h1_h2_fold_protocol.csv"
    )

    OUTPUT_DIR = (
        "/content/h10_results"
    )

    K_VALUES = [
        5,
        10,
        20,
    ]

    EPISODES = 2000

    SEED = 42

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device(
            "cuda"
        )

        print(
            "CUDA available: True"
        )

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    else:

        DEVICE = torch.device(
            "cpu"
        )

        print(
            "CUDA available: False"
        )

        print(
            "Using CPU."
        )

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    set_seed(
        SEED
    )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    print()

    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            JSON_PATH
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()

    print(
        "Class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    print()

    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            FOLDS_PATH
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # --------------------------------------------------------
    # Run H10
    # --------------------------------------------------------

    print()

    print("=" * 80)

    print(
        "STARTING H10"
    )

    print("=" * 80)

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=K_VALUES,
        seed=SEED,
        episodes=EPISODES,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )

CUDA available: True
GPU: Tesla T4

Loading clip data...
Loaded 11902 labelled clips from 103 students.

Class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Loading fold protocol...
Fold protocol contains 82 students.

STARTING H10

H10 OUTER FOLD 1
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

H10 meta-training (2000 episodes)
    Episode 100/2000 loss=2.4384 CE=0.9293 Proto=1.0855 SupCon=4.1658 Ord=0.2319 HD/D=-0.0000
    Episode 200/2000 loss=1.6851 CE=0.7586 Proto=1.0593 SupCon=4.1769 Ord=0.1414 HD/D=-0.0000
    Episode 300/2000 loss=1.3836 CE=0.8508 Proto=1.3475 SupCon=4.0717 Ord=0.2366 HD/D=0.0000
    Episode 400/2000 loss=1.3535 CE=0.3724 Proto=0.8138 SupCon=4.0431 Ord=0.0505 HD/D=0.0000
    Episode 500/2000 loss=1.2933 CE=0.

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

Completed fold 1

H10 OUTER FOLD 2
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

H10 meta-training (2000 episodes)
    Episode 100/2000 loss=2.6289 CE=0.9298 Proto=1.3545 SupCon=5.1748 Ord=0.1430 HD/D=0.5638
    Episode 200/2000 loss=1.6026 CE=0.8343 Proto=1.2725 SupCon=4.2531 Ord=0.1297 HD/D=0.4912
    Episode 300/2000 loss=1.4515 CE=0.9057 Proto=1.2588 SupCon=4.1471 Ord=0.1414 HD/D=0.0000
    Episode 400/2000 loss=1.2839 CE=0.4266 Proto=0.8423 SupCon=4.2016 Ord=0.0635 HD/D=0.0000
    Episode 500/2000 loss=1.2358 CE=1.1221 Proto=0.9510 SupCon=4.2010 Ord=0.1679 HD/D=0.0000
    Episode 600/2000 loss=1.2073 CE=0.8684 Proto=0.8525 SupCon=3.5415 Ord=0.1591 HD/D=0.0000
    Episode 700/2000 loss=1.1521 CE=0.7435 Proto=1.2118 SupCon=4.1817 Ord=0.1334 HD/D=0.0000
    Episode 800/2000 loss=1.0818 CE=0.2812 Proto=0.9967 SupCon=3.8923 Ord=0.0

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

Completed fold 2

H10 OUTER FOLD 3
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

H10 meta-training (2000 episodes)
    Episode 100/2000 loss=2.7665 CE=0.6461 Proto=1.0849 SupCon=4.9699 Ord=0.1159 HD/D=-0.0000
    Episode 200/2000 loss=1.5990 CE=0.8718 Proto=1.0575 SupCon=5.1303 Ord=0.1501 HD/D=0.0000
    Episode 300/2000 loss=1.6167 CE=0.4940 Proto=0.8034 SupCon=3.9954 Ord=0.0633 HD/D=-0.0000
    Episode 400/2000 loss=1.2982 CE=0.0523 Proto=0.3228 SupCon=4.1532 Ord=0.0013 HD/D=0.0000
    Episode 500/2000 loss=1.2792 CE=0.4022 Proto=0.9611 SupCon=4.3505 Ord=0.0495 HD/D=0.0000
    Episode 600/2000 loss=1.2508 CE=0.5501 Proto=1.1693 SupCon=4.0475 Ord=0.0968 HD/D=0.0000
    Episode 700/2000 loss=1.1927 CE=0.5060 Proto=1.0708 SupCon=4.2361 Ord=0.0744 HD/D=0.0000
    Episode 800/2000 loss=1.0839 CE=0.4736 Proto=1.0360 SupCon=4.0737 Ord=0

...

Completed fold 3

H10 OUTER FOLD 4
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

H10 meta-training (2000 episodes)
    Episode 100/2000 loss=2.4445 CE=0.3178 Proto=0.6392 SupCon=4.0198 Ord=0.0192 HD/D=0.0000
    Episode 200/2000 loss=1.6273 CE=1.0402 Proto=1.0611 SupCon=5.0423 Ord=0.1922 HD/D=-0.0000
    Episode 300/2000 loss=1.4618 CE=1.1381 Proto=1.3259 SupCon=5.7025 Ord=0.2881 HD/D=0.0000
    Episode 400/2000 loss=1.3083 CE=0.2058 Proto=0.7134 SupCon=3.5474 Ord=0.0221 HD/D=0.0000
    Episode 500/2000 loss=1.2988 CE=0.8210 Proto=1.3128 SupCon=3.8685 Ord=0.1683 HD/D=0.0000
    Episode 600/2000 loss=1.1802 CE=0.8327 Proto=0.5535 SupCon=3.7811 Ord=0.1012 HD/D=0.0000
    Episode 700/2000 loss=1.2142 CE=0.3032 Proto=0.5964 SupCon=3.8730 Ord=0.0418 HD/D=0.0000
    Episode 800/2000 loss=1.1686 CE=0.3150 Proto=1.0981 SupCon=3.8461 Ord=0.

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

Completed fold 4

H10 OUTER FOLD 5
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

H10 meta-training (2000 episodes)
    Episode 100/2000 loss=2.5937 CE=0.8804 Proto=1.3781 SupCon=5.0063 Ord=0.2115 HD/D=0.3829
    Episode 200/2000 loss=1.6501 CE=0.7489 Proto=1.0624 SupCon=4.5690 Ord=0.1093 HD/D=0.0000
    Episode 300/2000 loss=1.4133 CE=0.9143 Proto=1.3261 SupCon=4.1634 Ord=0.2077 HD/D=0.0000
    Episode 400/2000 loss=1.3557 CE=0.9292 Proto=1.3141 SupCon=4.1983 Ord=0.2908 HD/D=0.0000
    Episode 500/2000 loss=1.3153 CE=0.5288 Proto=1.1618 SupCon=4.3435 Ord=0.0918 HD/D=0.1344
    Episode 600/2000 loss=1.2592 CE=0.4035 Proto=0.8983 SupCon=4.3761 Ord=0.0664 HD/D=0.0000
    Episode 700/2000 loss=1.1819 CE=0.3720 Proto=0.7795 SupCon=3.5563 Ord=0.0535 HD/D=0.0000
    Episode 800/2000 loss=1.1812 CE=1.2617 Proto=0.9883 SupCon=3.8606 Ord=0.3

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics

Completed fold 5

H10: RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_hyper_accuracy  mean_prototype_accuracy  mean_prototype_balanced_accuracy  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1  mean_prototype_classes_available
           early       5          82              11016       0.589946     0.200215                0.450206              0.145313       0.315165     0.095457             0.589946                 0.677468                          0.336887                0.708270                         0.307927                0.205566                          1.902439
           early      10          82              10606       0.604610     0.190739                0.452452              0.145155       0.318375     0.090802             0.604610                 0.695814                          0.355228                0.710367  